## Dashboard Preparation: Loading Data and Libraries

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styling for plots
sns.set_theme(style='whitegrid')

### Data Loading and Preprocessing

Loading the necessary CSV files and preparing them for analysis. This involves converting `transaction_time` to datetime objects and merging merchant data.

In [12]:
# Load the datasets
ledger_df = pd.read_csv('ledger.csv')
gateway_df = pd.read_csv('gateway_export.csv')
merchants_df = pd.read_csv('merchants.csv')

# Ensure date formatting for ledger_df
ledger_df['transaction_time'] = pd.to_datetime(ledger_df['transaction_time'])
ledger_df['date_only'] = ledger_df['transaction_time'].dt.date

# Ensure date formatting for gateway_df
gateway_df['transaction_time'] = pd.to_datetime(gateway_df['transaction_time'])

# Merge ledger_df with merchants_df to get category information
merged_df = pd.merge(ledger_df, merchants_df, on='merchant_id', how='left')

print("Data loaded and preprocessed. Displaying head of merged_df:")
display(merged_df.head())

Data loaded and preprocessed. Displaying head of merged_df:


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score,date_only,merchant_name,category,region
0,TXN100000,350,16,2026-01-29 11:13:00,2999,Wallet,captured,85,2026-01-29,Merchant_016,bill_payment,West
1,TXN100001,287,27,2026-01-04 11:49:00,49,UPI,captured,79,2026-01-04,Merchant_027,ecommerce,North
2,TXN100002,122,11,2026-01-24 04:59:00,1499,Wallet,captured,22,2026-01-24,Merchant_011,ecommerce,North
3,TXN100003,92,22,2026-01-29 13:01:00,49,Netbanking,captured,100,2026-01-29,Merchant_022,travel,South
4,TXN100004,343,16,2026-01-30 13:51:00,99,UPI,captured,34,2026-01-30,Merchant_016,bill_payment,West


## Headline Layer: Scorecards

Calculating and displaying key performance indicators (KPIs) as defined:
1.  **Total GMV in INR**
2.  **Overall Success Rate** (transactions with status 'captured')
3.  **Reconciliation Match Rate**
4.  **Platform-wide Chargeback Ratio**

In [13]:
# 1. Total GMV in INR
total_gmv_inr = ledger_df['amount_inr'].sum()

# Calculate total transaction count for rates
total_tx_count = len(ledger_df)

# 2. Overall Success Rate (based on 'captured' status)
success_tx_count = ledger_df[ledger_df['status'] == 'captured'].shape[0]
overall_success_rate = (success_tx_count / total_tx_count) * 100 if total_tx_count > 0 else 0

# 3. Reconciliation Match Rate
# Merge ledger and gateway DFs on transaction_id, amount_inr, and status
matched_transactions = pd.merge(
    ledger_df,
    gateway_df,
    on=['transaction_id', 'amount_inr', 'status'],
    how='inner'
)
match_rate = (len(matched_transactions) / total_tx_count) * 100 if total_tx_count > 0 else 0

# 4. Chargeback Ratio (platform-wide, count-based)
cb_tx_count = ledger_df[ledger_df['status'] == 'chargeback'].shape[0]
chargeback_ratio = (cb_tx_count / total_tx_count) * 100 if total_tx_count > 0 else 0

print("--- HEADLINE SCORECARDS ---")
print(f"Total GMV (INR): ₹{total_gmv_inr:,.2f}")
print(f"Overall Success Rate: {overall_success_rate:.2f}%")
print(f"Reconciliation Match Rate: {match_rate:.2f}%")
print(f"Chargeback Ratio: {chargeback_ratio:.2f}%")

--- HEADLINE SCORECARDS ---
Total GMV (INR): ₹382,603.00
Overall Success Rate: 85.56%
Reconciliation Match Rate: 90.49%
Chargeback Ratio: 5.12%


## Trends Layer: Daily GMV and Chargeback Count Over 30 Days

Preparing data and generating time-series charts for daily GMV and daily chargeback count to observe trends over a 30-day window.

In [14]:
# Calculate daily GMV
daily_gmv = ledger_df.groupby('date_only')['amount_inr'].sum().reset_index(name='daily_gmv')

# Calculate daily chargeback count
daily_cb = ledger_df[ledger_df['status'] == 'chargeback'].groupby('date_only').size().reset_index(name='daily_cb_count')

# Merge daily GMV and chargebacks, filling missing chargeback counts with 0
daily_trends = pd.merge(daily_gmv, daily_cb, on='date_only', how='left').fillna(0)

# Filter for the last 30 days
if not daily_trends.empty:
    max_date = daily_trends['date_only'].max()
    min_date = max_date - pd.Timedelta(days=29) # 30 days including max_date
    daily_trends_30d = daily_trends[(daily_trends['date_only'] >= min_date) & (daily_trends['date_only'] <= max_date)]
else:
    daily_trends_30d = pd.DataFrame(columns=['date_only', 'daily_gmv', 'daily_cb_count'])

print("Daily trends data prepared. Displaying head of daily_trends_30d:")
display(daily_trends_30d.head())

Daily trends data prepared. Displaying head of daily_trends_30d:


,date_only,daily_gmv,daily_cb_count
0,2026-01-01,11982,0.0
1,2026-01-02,13430,0.0
2,2026-01-03,4129,0.0
3,2026-01-04,10688,1.0
4,2026-01-05,22030,1.0


In [15]:
if not daily_trends_30d.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Plot Daily GMV
    sns.lineplot(ax=ax1, x='date_only', y='daily_gmv', data=daily_trends_30d, marker='o', linewidth=2, color='#1f77b4')
    ax1.set_title('30-Day Daily GMV Trend (INR)', fontsize=14, pad=10)
    ax1.set_ylabel('GMV (INR)')
    ax1.ticklabel_format(style='plain', axis='y') # Prevent scientific notation
    ax1.grid(True, linestyle='--', alpha=0.7)

    # Plot Daily Chargeback Count
    sns.lineplot(ax=ax2, x='date_only', y='daily_cb_count', data=daily_trends_30d, marker='s', linewidth=2, color='#d62728')
    ax2.set_title('30-Day Daily Chargeback Count', fontsize=14, pad=10)
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Chargeback Count')
    ax2.grid(True, linestyle='--', alpha=0.7)
    ax2.set_xticks(daily_trends_30d['date_only'])
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('layer2_trends.png', dpi=300)
    plt.close()
    print("Daily trends chart saved as 'layer2_trends.png'.")
else:
    print("No data available to plot daily trends.")

Daily trends chart saved as 'layer2_trends.png'.


## Breakdown Layer: GMV by Payment Method and Category

Aggregating GMV by payment method and merchant category, and visualizing these breakdowns with bar charts.

In [16]:
# Calculate GMV by payment method
gmv_by_method = merged_df.groupby('payment_method')['amount_inr'].sum().sort_values(ascending=False).reset_index()

# Calculate GMV by category
gmv_by_category = merged_df.groupby('category')['amount_inr'].sum().sort_values(ascending=False).reset_index()

print("GMV by Payment Method:")
display(gmv_by_method)

print("\nGMV by Category:")
display(gmv_by_category)

GMV by Payment Method:


,payment_method,amount_inr
0,UPI,172274
1,Card,102429
2,Wallet,71304
3,Netbanking,36596



GMV by Category:


,category,amount_inr
0,ecommerce,79896
1,travel,75250
2,grocery,71936
3,food_delivery,57205
4,entertainment,56887
5,bill_payment,26304
6,recharge,15125


In [17]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart for GMV by Payment Method
sns.barplot(ax=ax1, data=gmv_by_method, x='payment_method', y='amount_inr', palette='Blues_d', hue='payment_method', legend=False)
ax1.set_title('GMV by Payment Method', fontsize=16)
ax1.set_xlabel('Payment Method', fontsize=12)
ax1.set_ylabel('Total GMV (INR)', fontsize=12)
ax1.ticklabel_format(style='plain', axis='y')

# Bar chart for GMV by Category
sns.barplot(ax=ax2, data=gmv_by_category, x='category', y='amount_inr', palette='Greens_d', hue='category', legend=False)
ax2.set_title('GMV by Category', fontsize=16)
ax2.set_xlabel('Category', fontsize=12)
ax2.set_ylabel('Total GMV (INR)', fontsize=12)
ax2.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.savefig('layer3_breakdown.png', dpi=300)
plt.close()
print("GMV breakdown charts saved as 'layer3_breakdown.png'.")

GMV breakdown charts saved as 'layer3_breakdown.png'.


## Details Layer: Top 10 Merchants Table

Calculating transaction statistics for each merchant, identifying the top 10 by transaction count, and highlighting merchants with a chargeback ratio exceeding 1%. The table will be rendered as an image.

In [18]:
# Calculate merchant-level statistics
merchant_stats = ledger_df.groupby('merchant_id').agg(
    total_tx=('transaction_id', 'count'),
    cb_tx=('status', lambda x: (x == 'chargeback').sum())
).reset_index()

# Calculate per-merchant chargeback ratio
merchant_stats['chargeback_ratio'] = (merchant_stats['cb_tx'] / merchant_stats['total_tx']) * 100
merchant_stats['high_risk_flag'] = merchant_stats['chargeback_ratio'] > 1.0

# Extract top 10 merchants by transaction count
top10_merchants = merchant_stats.sort_values(by='total_tx', ascending=False).head(10).copy()
top10_merchants['chargeback_ratio_str'] = top10_merchants['chargeback_ratio'].map('{:.2f}%'.format)

print("Top 10 merchants data prepared. Displaying head:")
display(top10_merchants)

Top 10 merchants data prepared. Displaying head:


,merchant_id,total_tx,cb_tx,chargeback_ratio,high_risk_flag,chargeback_ratio_str
15,16,20,0,0.000000,False,0.00%
28,29,19,3,15.789474,True,15.79%
36,37,19,1,5.263158,True,5.26%
8,9,18,0,0.000000,False,0.00%
2,3,17,1,5.882353,True,5.88%
24,25,17,1,5.882353,True,5.88%
29,30,17,0,0.000000,False,0.00%
7,8,16,1,6.250000,True,6.25%
35,36,16,1,6.250000,True,6.25%
26,27,16,3,18.750000,True,18.75%


In [23]:
# Prepare data for table visualization
table_matrix = top10_merchants[['merchant_id', 'total_tx', 'cb_tx', 'chargeback_ratio_str', 'high_risk_flag']].values
headers = ['Merchant ID', 'Total Tx Count', 'Chargebacks', 'CB Ratio', 'High Risk (>1%)']

# Create a matplotlib figure and axis for the table
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off') # Hide axes

# Render the table
rendered_table = ax.table(cellText=table_matrix, colLabels=headers, loc='center', cellLoc='center')
rendered_table.auto_set_font_size(False)
rendered_table.set_fontsize(10)
rendered_table.scale(1.2, 1.3)

# Apply conditional highlighting for high-risk merchants
for row_idx, row in enumerate(top10_merchants.itertuples()):
    if row.high_risk_flag:
        for col_idx in range(len(headers)):
            rendered_table[(row_idx + 1, col_idx)].set_facecolor('#ffcccc') # Light red background for high risk

plt.title('Top 10 Merchants by Transaction Count (High Risk Flagged)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('layer4_details_table.png', dpi=300, bbox_inches='tight')
plt.close()
print("Top 10 merchants table saved as 'layer4_details_table.png'.")

Top 10 merchants table saved as 'layer4_details_table.png'.


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styling
sns.set_theme(style='whitegrid')

# ==========================================
# 0. LOAD DATA
# ==========================================
ledger_df = pd.read_csv('ledger.csv')
gateway_df = pd.read_csv('gateway_export.csv')
merchants_df = pd.read_csv('merchants.csv')

# Ensure date formatting
ledger_df['transaction_time'] = pd.to_datetime(ledger_df['transaction_time'])
ledger_df['date_only'] = ledger_df['transaction_time'].dt.date

gateway_df['transaction_time'] = pd.to_datetime(gateway_df['transaction_time'])

# Merge merchant metadata for breakdown
merged_df = pd.merge(ledger_df, merchants_df, on='merchant_id', how='left')


# ==========================================
# 1. HEADLINE LAYER (Scorecards)
# ==========================================
total_gmv_inr = ledger_df['amount_inr'].sum()
total_tx_count = len(ledger_df)

# Success Rate
success_tx_count = ledger_df[ledger_df['status'] == 'captured'].shape[0]
overall_success_rate = (success_tx_count / total_tx_count) * 100 if total_tx_count > 0 else 0

# Match Rate (Identical transaction_id, amount_inr, and status in both files)
matched_df = pd.merge(
    ledger_df,
    gateway_df,
    on=['transaction_id', 'amount_inr', 'status'],
    how='inner'
)
match_rate = (len(matched_df) / total_tx_count) * 100 if total_tx_count > 0 else 0

# Chargeback Ratio (Platform-wide count-based)
cb_tx_count = ledger_df[ledger_df['status'] == 'chargeback'].shape[0]
chargeback_ratio = (cb_tx_count / total_tx_count) * 100 if total_tx_count > 0 else 0

print("--- HEADLINE SCORECARDS ---")
print(f"Total GMV (INR): ₹{total_gmv_inr:,.2f}")
print(f"Overall Success Rate: {overall_success_rate:.2f}%")
print(f"Reconciliation Match Rate: {match_rate:.2f}%")
print(f"Chargeback Ratio: {chargeback_ratio:.2f}%")


# ==========================================
# 2. TRENDS LAYER (30-Day Time Series)
# ==========================================
daily_gmv = ledger_df.groupby('date_only')['amount_inr'].sum().reset_index(name='daily_gmv')
daily_cb = ledger_df[ledger_df['status'] == 'chargeback'].groupby('date_only').size().reset_index(name='daily_cb_count')

daily_trends = pd.merge(daily_gmv, daily_cb, on='date_only', how='left').fillna(0)

# Filter for the 30-day window
max_date = daily_trends['date_only'].max()
min_date = max_date - pd.Timedelta(days=29)
daily_trends_30d = daily_trends[daily_trends['date_only'] >= min_date]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# GMV Line Chart
ax1.plot(daily_trends_30d['date_only'], daily_trends_30d['daily_gmv'], marker='o', linewidth=2, color='#1f77b4')
ax1.set_title('30-Day Daily GMV Trend (INR)', fontsize=14, pad=10)
ax1.set_ylabel('GMV (INR)')

# Chargeback Count Line Chart
ax2.plot(daily_trends_30d['date_only'], daily_trends_30d['daily_cb_count'], marker='s', linewidth=2, color='#d62728')
ax2.set_title('30-Day Daily Chargeback Count', fontsize=14, pad=10)
ax2.set_xlabel('Date')
ax2.set_ylabel('Chargeback Count')

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('layer2_trends.png', dpi=300)
plt.close()


# ==========================================
# 3. BREAKDOWN LAYER (Bar Charts)
# ==========================================
gmv_by_method = merged_df.groupby('payment_method')['amount_inr'].sum().reset_index()
gmv_by_category = merged_df.groupby('category')['amount_inr'].sum().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(ax=ax1, data=gmv_by_method, x='payment_method', y='amount_inr', palette='Blues_d', hue='payment_method', legend=False)
ax1.set_title('GMV by Payment Method', fontsize=12)
ax1.set_xlabel('Payment Method')
ax1.set_ylabel('Total GMV (INR)')

sns.barplot(ax=ax2, data=gmv_by_category, x='category', y='amount_inr', palette='Greens_d', hue='category', legend=False)
ax2.set_title('GMV by Category', fontsize=12)
ax2.set_xlabel('Category')
ax2.set_ylabel('Total GMV (INR)')

plt.tight_layout()
plt.savefig('layer3_breakdown.png', dpi=300)
plt.close()


# ==========================================
# 4. DETAILS LAYER (Top 10 Merchants Image Table)
# ==========================================
merchant_stats = ledger_df.groupby('merchant_id').agg(
    total_tx=('transaction_id', 'count'),
    cb_tx=('status', lambda x: (x == 'chargeback').sum())
).reset_index()

merchant_stats['chargeback_ratio'] = (merchant_stats['cb_tx'] / merchant_stats['total_tx']) * 100
merchant_stats['high_risk_flag'] = merchant_stats['chargeback_ratio'] > 1.0

# Extract top 10 merchants by transaction count
top10_merchants = merchant_stats.sort_values(by='total_tx', ascending=False).head(10).copy()
top10_merchants['chargeback_ratio_str'] = top10_merchants['chargeback_ratio'].map('{:.2f}%'.format)

# Format table data matrix
table_matrix = top10_merchants[['merchant_id', 'total_tx', 'cb_tx', 'chargeback_ratio_str', 'high_risk_flag']].values
headers = ['Merchant ID', 'Total Tx Count', 'Chargebacks', 'CB Ratio', 'High Risk (>1%)']

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

rendered_table = ax.table(cellText=table_matrix, colLabels=headers, loc='center', cellLoc='center')
rendered_table.auto_set_font_size(False)
rendered_table.set_fontsize(10) # Corrected from set_font_size to set_fontsize
rendered_table.scale(1.2, 1.3)

# Highlight high-risk rows
for row_idx, row in enumerate(top10_merchants.itertuples()):
    if row.high_risk_flag:
        for col_idx in range(len(headers)):
            rendered_table[(row_idx + 1, col_idx)].set_facecolor('#ffcccc')

plt.title('Top 10 Merchants by Transaction Count (High Risk Flagged)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('layer4_details_table.png', dpi=300, bbox_inches='tight')
plt.close()

print("Dashboard graphics generated and saved successfully.")

--- HEADLINE SCORECARDS ---
Total GMV (INR): ₹382,603.00
Overall Success Rate: 85.56%
Reconciliation Match Rate: 90.49%
Chargeback Ratio: 5.12%
Dashboard graphics generated and saved successfully.
